# 02 · Geração fundamentada: JSON válido, citação da fonte e recusa

Item 04 da rubrica. Toda resposta sai no formato `{resposta, fonte, consta}`, validado com Pydantic (`src/esquema.py`). Se o JSON vier inválido, o sistema pede de novo uma vez; se continuar inválido, recusa.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import pandas as pd
pd.set_option('display.max_colwidth', 120)
from src import config
from src.busca import Indice
from src.avaliar import carregar_perguntas
indice = Indice()
perguntas = carregar_perguntas()
indice.pre_calcular([p['pergunta'] for p in perguntas])  # vetoriza as perguntas uma vez (cache)
print(len(indice.trechos), 'trechos |', len(perguntas), 'perguntas | LLM:', config.MODELO_LLM, '| embeddings:', config.MODELO_EMB)
from src.pipelines import sem_contexto, rag, agente
P = {p['id']: p['pergunta'] for p in perguntas}
CAMPOS = ('resposta', 'fonte', 'consta', 'motivo', 'evidencias', 'passos', 'json_valido', 'tentativas_json', 'latencia_s')
def mostrar(r): print(json.dumps({k: r[k] for k in CAMPOS if k in r}, ensure_ascii=False, indent=1))

32 trechos | 36 perguntas | LLM: qwen2.5:3b | embeddings: bge-m3


### 1. Resposta com citação (RAG)

In [2]:
print(P['q01']); mostrar(rag(P['q01'], indice))

Qual a distância lateral mínima que um carro deve manter ao ultrapassar uma bicicleta?


{
 "resposta": "O carro deve manter uma distância lateral de um metro e meio ao ultrapassar uma bicicleta.",
 "fonte": "ctb-art201",
 "consta": true,
 "evidencias": [
  [
   "ctb-art201",
   0.679
  ],
  [
   "ctb-art220-xiii",
   0.576
  ],
  [
   "ctb-art58",
   0.574
  ]
 ],
 "json_valido": true,
 "tentativas_json": 1,
 "latencia_s": 3.87
}


In [3]:
print(P['q09']); mostrar(rag(P['q09'], indice))

Qual a potência máxima do motor de uma bicicleta elétrica?


{
 "resposta": "A potência máxima do motor de uma bicicleta elétrica é de até 1000 W (mil watts).",
 "fonte": "r996-art2-iii",
 "consta": true,
 "evidencias": [
  [
   "r996-art2-iii",
   0.708
  ],
  [
   "r996-art2-p1-p4",
   0.634
  ],
  [
   "r996-art2-iv-p6",
   0.618
  ]
 ],
 "json_valido": true,
 "tentativas_json": 1,
 "latencia_s": 1.12
}


### 2. Recusa: a pergunta não tem resposta na base

In [4]:
print(P['q26']); mostrar(rag(P['q26'], indice))

Quanto custa, em média, um seguro para bicicleta elétrica?
{
 "resposta": "Não consta na base de conhecimento do CicloAlerta.",
 "fonte": "",
 "consta": false,
 "motivo": "abaixo_do_limiar",
 "evidencias": [
  [
   "r996-art2-p1-p4",
   0.557
  ],
  [
   "r996-art4",
   0.536
  ],
  [
   "r996-art2-iii",
   0.532
  ]
 ],
 "json_valido": true,
 "tentativas_json": 0,
 "latencia_s": 0.0
}


In [5]:
print(P['q29']); mostrar(rag(P['q29'], indice))

É obrigatório usar capacete para andar de bicicleta comum?
{
 "resposta": "Não consta na base de conhecimento do CicloAlerta.",
 "fonte": "",
 "consta": false,
 "motivo": "abaixo_do_limiar",
 "evidencias": [
  [
   "ctb-art58",
   0.527
  ],
  [
   "ctb-art105-vi",
   0.518
  ],
  [
   "r996-art4",
   0.515
  ]
 ],
 "json_valido": true,
 "tentativas_json": 0,
 "latencia_s": 0.0
}


### 3. A mesma pergunta sem contexto: o modelo inventa

Sem a base, o modelo tende a responder com confiança e a citar fontes que não foram dadas a ele. É isso que o placar mede como alucinação.

In [6]:
print(P['q29']); mostrar(sem_contexto(P['q29']))

É obrigatório usar capacete para andar de bicicleta comum?


{
 "resposta": "Não, não é obrigatório usar capacete para andar de bicicleta comum em São Paulo.",
 "fonte": "consta=false",
 "consta": false,
 "evidencias": [],
 "json_valido": true,
 "tentativas_json": 1,
 "latencia_s": 0.98
}


In [7]:
print(P['q28']); mostrar(sem_contexto(P['q28']))

Qual o valor em reais da multa para quem estaciona o carro na ciclofaixa?


{
 "resposta": "A multa para quem estaciona carro na ciclofaixa em São Paulo é de R$ 350,00.",
 "fonte": "CicloAlerta",
 "consta": true,
 "evidencias": [],
 "json_valido": true,
 "tentativas_json": 1,
 "latencia_s": 1.11
}


### 4. Agente: escolhe a ferramenta e verifica a resposta

In [8]:
print(P['q33']); mostrar(agente(P['q33'], indice))

Está chovendo agora na Avenida Paulista?


{
 "resposta": "Sim, está chovendo agora na Avenida Paulista com 0.3 mm de precipitação.",
 "fonte": "open-meteo:paulista",
 "consta": true,
 "evidencias": [
  [
   "open-meteo:paulista",
   null
  ]
 ],
 "passos": [
  "roteador -> clima (paulista)",
  "verificação ok"
 ],
 "json_valido": true,
 "tentativas_json": 1,
 "latencia_s": 2.79
}


In [9]:
print(P['q22']); mostrar(agente(P['q22'], indice))

Quando o CicloAlerta emite um alerta crítico?


{
 "resposta": "O CicloAlerta emite um alerta crítico quando há um perigo de severidade alta com chuva prevista igual ou acima de 1,0 mm em alguma hora.",
 "fonte": "faq-alertas",
 "consta": true,
 "evidencias": [
  [
   "faq-alertas",
   0.733
  ],
  [
   "faq-chuva-seguranca",
   0.691
  ],
  [
   "faq-reportar",
   0.651
  ]
 ],
 "passos": [
  "roteador -> base (None)",
  "verificação ok"
 ],
 "json_valido": true,
 "tentativas_json": 1,
 "latencia_s": 1.74
}


### 5. Validação do formato

O Pydantic rejeita uma saída fora do contrato (exemplo artificial):

In [10]:
from src.esquema import Resposta
from pydantic import ValidationError
exemplos = [json.dumps({'resposta': '1,50 m', 'fonte': 'ctb-art201', 'consta': True}), json.dumps({'resposta': '1,50 m'})]
for bruto in exemplos:
    try: print('válido   ->', Resposta.model_validate_json(bruto))
    except ValidationError as e: print('inválido ->', bruto, '| erro:', e.errors()[0]['type'], e.errors()[0]['loc'])

válido   -> resposta='1,50 m' fonte='ctb-art201' consta=True
inválido -> {"resposta": "1,50 m"} | erro: missing ('fonte',)
